In [0]:
from pyspark.sql import SparkSession
import requests
from datetime import datetime

# Start Spark session
spark = SparkSession.builder.getOrCreate()

API_KEY = ""

#  List of cities
cities = ["Nairobi", "Mombasa", "Kisumu", "Kampala", "Dar es Salaam"]

url = "https://api.openweathermap.org/data/2.5/weather"

records = []

# Loop through cities
for city in cities:
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()

        records.append({
            "city": data["name"],
            "temperature": data["main"]["temp"],
            "humidity": data["main"]["humidity"],
            "weather_description": data["weather"][0]["description"],
            "timestamp": datetime.now()
        })

    else:
        print(f"Failed for {city}: {response.status_code}")

# Convert to DataFrame
if records:
    df = spark.createDataFrame(records)

    # Write to table
    df.write.mode("append").saveAsTable("openweather.openweather_b.openweather_landing")

    print("ll city data inserted successfully!")
else:
    print("No data collected")

In [0]:
%sql
CREATE OR REPLACE VIEW openweather.openweather_b.openweather_vw AS
SELECT
  city AS city,
  humidity AS humidity,
  temperature AS temperature,
  timestamp AS timestamp,
  weather_description AS weather_description
FROM
  openweather.openweather_b.openweather_landing

In [0]:
spark.sql("""
COMMENT ON VIEW openweather.openweather_b.openweather_vw IS 'Weather data for East African cities'
""")
spark.sql("""
COMMENT ON COLUMN openweather.openweather_b.openweather_vw.city IS 'Name of the city'
""")
spark.sql("""
COMMENT ON COLUMN openweather.openweather_b.openweather_vw.humidity IS 'Humidity percentage'
""")
spark.sql("""
COMMENT ON COLUMN openweather.openweather_b.openweather_vw.temperature IS 'Temperature in Celsius'
""")
spark.sql("""
COMMENT ON COLUMN openweather.openweather_b.openweather_vw.timestamp IS 'Timestamp of data retrieval'
""")
spark.sql("""
COMMENT ON COLUMN openweather.openweather_b.openweather_vw.weather_description IS 'Weather description'
""")